# Transfer entropy between stocks

The transfer-entropy (TE) stage of the thesis that follows
[entropy-gate-pipeline](https://github.com/Gaire-commits/entropy-gate-pipeline). Question: does a
stock's recent 5-minute move tell us about another stock's next move, beyond the second stock's own
recent moves (including its own 30-minute reversal) and beyond the market?

**Same data and conventions as the thesis**
- The cached **5-minute SIP bars** on Google Drive (503 S&P 500 stocks, 2016-01-04 to 2026-10-12).
  Nothing is downloaded.
- A fixed 5-minute grid per trading day. Missing bars stay as gaps, and no return spans the night.
- **Market-neutral.** The market is the equal-weighted mean return across the universe at each bar,
  regressed out.
- **Inference by day.** Whole days go to train, validation or test (the last 20% of days is the
  hold-out, about Aug 2024 to Oct 2026), and intervals resample whole days.
- Synthetic controls first, then the proposal's pairs, with false-discovery control across them.

TE values are in nats. These runs are exploratory: freeze the settings in a config before any
result is quoted in the thesis.

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/Transfer-entropy-"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Gaire-commits/Transfer-entropy-.git {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
else:  # running locally from a clone
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO)

# Forget any learned_te code loaded earlier in this session, so re-running this
# cell after the pull above always uses the latest version (no restart needed).
for name in [m for m in sys.modules if m == "learned_te" or m.startswith("learned_te.")]:
    del sys.modules[name]

import warnings
warnings.filterwarnings("ignore")  # library noise (sklearn)
warnings.filterwarnings("default", module="learned_te")  # but show e.g. skipped days
import pandas as pd
from learned_te import Config, report, simulate, SCENARIOS
from learned_te.market import (THESIS_SIP_DIR, completeness, day_sessions, load_cached_closes,
                               market_returns, session_grid)
from learned_te.multiple import benjamini_hochberg

# Thesis folders on Drive (TE_DATA_DIR / TE_OUT_DIR override them, e.g. for a local test).
DATA_DIR = os.environ.get("TE_DATA_DIR", THESIS_SIP_DIR)
OUT_DIR = os.environ.get("TE_OUT_DIR", "/content/drive/MyDrive/entropy-gate-pipeline-data/outputs/transfer_entropy")
print("ready:", REPO)
print("SIP bars:", DATA_DIR, "| found:", os.path.isdir(os.path.join(DATA_DIR, "5Min")))

## 2. Controls: simulated systems with known answers

Before real data, the method must find a planted link and reject the two ways noise fakes one.
Only `coupled` has a real X→Y link; `proxy_leak` and `shared_noise` pass the significance test but
should be flagged by the diagnostics.

In [ ]:
def row(rep, label):
    f, r = rep["forward"], rep["reverse"]
    return {"analysis": label, "T_eff": round(f.te_eff, 5),
            "95% CI": f"[{f.ci[0]:+.5f}, {f.ci[1]:+.5f}]", "p": round(f.p_value, 4),
            "T_eff reverse": round(r.te_eff, 5), "verdict": rep["verdict"]}

controls = []
for name, (truth, _) in SCENARIOS.items():
    rep = report(simulate(name, seed=1), Config(n_surrogates=100, n_bootstrap=300))
    controls.append(row(rep, f"{name} ({'link' if truth else 'no link'})"))
pd.DataFrame(controls).set_index("analysis")

## 3. The thesis data

The market return reads every cached stock once (a few minutes), then is saved to the outputs folder
on Drive and reused.

In [ ]:
PAIRS = [("NVDA", "AMD"), ("V", "MA"), ("MSFT", "GOOGL")]  # from the proposal
symbols = sorted({s for pair in PAIRS for s in pair})

closes = load_cached_closes(DATA_DIR, symbols)
print("bars per symbol on the grid (share of slots filled):")
print(completeness(session_grid(closes)).round(4).to_string())

market = market_returns(DATA_DIR, cache_file=os.path.join(OUT_DIR, "market_5min.parquet"))
print(f"market: {market.notna().sum():,} bars,", market.index[0].date(), "to", market.index[-1].date())

## 4. The proposal's pairs, both directions

History: k = 6 bars (30 minutes) of the target's own returns, so its own reversal is already in the
baseline, and l = 6 bars of the source. Each direction gets the full diagnostics. A direction counts
only if it survives false-discovery control across all directions tested **and** passes every leak
check. Each direction takes a few minutes.

In [ ]:
cfg = Config(k=6, l=6, ref_cols=(2,), n_surrogates=200, n_bootstrap=500)

rows = []
for a, b in PAIRS:
    for x, y in [(a, b), (b, a)]:
        sessions = day_sessions(closes, x=x, y=y, market=market)
        rep = report(sessions, cfg)
        rows.append(row(rep, f"{x} -> {y}"))
        print(f"{x} -> {y}: T_eff {rows[-1]['T_eff']:+.5f}, p {rows[-1]['p']}, {rep['verdict']}")

results = pd.DataFrame(rows).set_index("analysis")
results["passes FDR (q=0.05)"] = benjamini_hochberg(results["p"], q=0.05)
results["counts"] = results["passes FDR (q=0.05)"] & (results["verdict"] == "directed X->Y transfer supported")
results

## 5. Save

**Reading the result**
- *counts* = True: survives false-discovery control and every leak check. This is evidence of
  directed predictive information, not causation, and like the reversal it says nothing about
  trading profit (costs are reported separately, never as the verdict).
- *dependence present, direction unresolved*: significant but a leak check failed, which is the
  signature of shared noise or a noisy proxy.
- False-discovery control (Benjamini-Hochberg, q = 0.05) over 6 directions: the smallest p-value
  must be under 0.05/6 = 0.008 for anything to pass, the next under 0.017, and so on. The null uses
  200 surrogates so the smallest possible p (1/201 = 0.005) can clear that bar. Passing this control
  alone does not make a direction count; it must also pass every leak check.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
pd.concat([pd.DataFrame(controls).set_index("analysis"), results]).to_csv(os.path.join(OUT_DIR, "pairs.csv"))
print("saved", os.path.join(OUT_DIR, "pairs.csv"))